# term

> A client for gateway-hosted terminals: REST lifecycle plus one websocket attachment

In [ ]:
#| default_exp term

[rustygate](https://github.com/AnswerDotAI/rustygate) hosts terminals as siblings of kernels (ptys managed by [ptymini](https://github.com/AnswerDotAI/ptymini), exposed at `/api/terminals`), so a client app can put a real shell next to its kernel — same machine, same container, same filesystem. `JupyAsyncTerminalClient` is that surface from the client side, shaped like `JupyAsyncKernelClient`: the same `KernelApi` HTTP plumbing for lifecycle, and one websocket for the byte stream. The ws contract is jupygate's: binary frames are pty bytes verbatim in both directions; text frames are JSON control — the server sends `setup` on accept (any replayed scrollback follows as binary), `gap` when this client fell behind the replay ring, and `eof` (with the exit code) when the pty dies; the client sends `set_size`.

In [ ]:
#| export
import json, websockets
from contextlib import suppress
from fastcore.basics import patch
from jupyasyncclient.core import KernelApi, _join_url

In [ ]:
import asyncio, os, time
from fastcore.test import test_eq
from rustygate.tools import start_gateway


## Lifecycle

Construction mirrors `JupyAsyncKernelClient`: a gateway URL, an optional `name` to attach to an existing terminal (the parallel of passing a `kernel_id`), and the shared auth/timeout/http knobs from `KernelApi`. `start_terminal` POSTs a new terminal — the gateway's creation options (`argv`, `cwd`, `env`, `appendenv`, `rc`, `rows`, `cols`, `username`) pass through as given — and binds the returned name, the stable handle everything else uses.

In [ ]:
#| export
class JupyAsyncTerminalClient(KernelApi):
    "One gateway terminal: REST lifecycle over the shared `KernelApi` plumbing, plus one ws attachment."
    def __init__(self, base_url, name=None, token=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.name,self._ws = name,None

    def _tpath(self, name=''): return f'/api/terminals/{name}' if name else '/api/terminals'
    async def term_request(self, method, name='', **kwargs): return await self._request(method, self._tpath(name), **kwargs)
    async def list_terminals(self): return await self.term_request('GET')

    async def start_terminal(self, **kw):
        "Create a terminal (gateway creation options pass through) and bind its name."
        model = await self.term_request('POST', json=kw)
        self.name = model['name']
        return model

    async def shutdown_terminal(self): return await self.term_request('DELETE', self.name)

A live gateway to demonstrate against, and the deterministic shell the jupygate and ptymini pages use — `--norc --noprofile` with a fixed prompt, so output is predictable:

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')
g = start_gateway()
tc = JupyAsyncTerminalClient(g.url)
test_eq(await tc.list_terminals(), [])
model = await tc.start_terminal(argv=BASH, env=BENV)
test_eq((await tc.list_terminals())[0]['name'], tc.name)
model

## The channel

`connect` opens the terminal's websocket and returns the `setup` frame; whatever scrollback the terminal has already produced follows as binary. From there the surface is four verbs: `write` sends keystrokes (bytes, verbatim), `resize` sends the `set_size` control frame, `frames` is the incoming stream — pty output as `bytes`, control frames as parsed dicts — and `aclose` shuts the ws and any owned http client. `frames` is a plain async generator: hold one and drain it; two competing iterators would race for the same socket.

In [ ]:
#| export
@patch
async def connect(self:JupyAsyncTerminalClient):
    "Open the terminal's ws channel and return the `setup` frame; replayed scrollback follows as binary."
    params = dict(token=self.token) if self.token else None
    url = _join_url(self.base_url, self._tpath(self.name)+'/channel', ws=True, params=params)
    self._ws = await websockets.connect(url, ssl=self._ws_ssl(url), max_size=None)
    return json.loads(await self._ws.recv())

@patch
async def write(self:JupyAsyncTerminalClient, data:bytes): await self._ws.send(data)

@patch
async def resize(self:JupyAsyncTerminalClient, rows:int, cols:int):
    await self._ws.send(json.dumps(dict(type='set_size', rows=rows, cols=cols)))

@patch
async def frames(self:JupyAsyncTerminalClient):
    "Incoming frames: pty output as `bytes`, control frames as parsed dicts. Ends when the ws closes."
    with suppress(websockets.ConnectionClosed):
        async for frame in self._ws: yield frame if isinstance(frame, bytes) else json.loads(frame)

@patch
async def aclose(self:JupyAsyncTerminalClient):
    if self._ws is not None:
        with suppress(Exception): await self._ws.close()
        self._ws = None
    await self.aclose_http()

Type a command, read the echo and the output back. `read_until` accumulates binary frames until a pattern shows up — containment with a timeout, never exact frames, because pty chunking is timing-dependent by nature:

In [ ]:
async def read_until(fr, pat:bytes, timeout=10.0)->bytes:
    "Accumulate binary frames from async iterator `fr` until `pat` appears (control frames are skipped)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        item = await asyncio.wait_for(anext(fr), end - time.monotonic())
        if isinstance(item, bytes): buf += item
    return buf

setup = await tc.connect()
test_eq(setup['type'], 'setup')
fr = tc.frames()
await tc.write(b'echo wired $((6*7))\n')
out = await read_until(fr, b'wired 42')
out[-20:]

Reattach is the point of named terminals: a second client connecting to the same name gets `setup` and then the replayed scrollback — output of a command that ran before this client existed. A browser refresh, or an app restart, sees exactly this:

In [ ]:
tc2 = JupyAsyncTerminalClient(g.url, name=tc.name)
test_eq((await tc2.connect())['type'], 'setup')
fr2 = tc2.frames()
replay = await read_until(fr2, b'wired 42')
b'echo wired' in replay, b'wired 42' in replay

`resize` is verified in-band, like everything on a terminal — the shell reports its own new size. Then shutting the terminal down (either client can: DELETE is by name) ends every attachment with the `eof` control frame, which carries the pty's exit code:

In [ ]:
await tc.resize(50, 120)
await tc.write(b'stty size\n')
assert b'50 120' in await read_until(fr, b'50 120')
await tc2.shutdown_terminal()
item = await asyncio.wait_for(anext(fr), 10)
while isinstance(item, bytes): item = await asyncio.wait_for(anext(fr), 10)  # drain final output; the dict is the eof
test_eq(item['type'], 'eof')
test_eq(await tc.list_terminals(), [])
item

In [ ]:
#| hide
await tc.aclose()
await tc2.aclose()
g.stop()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()